# Лабораторная №1 курса "Синтез Речи"

## Импорты и константы

In [9]:
import re
import regex
import json
import pandas as pd
from tqdm import tqdm
from collections import Counter

import emoji

In [10]:
RUSLAN_METADATA_PATH = "../../data/metadata_RUSLAN_22200.csv"
RANDOM_STATE = 42

## EDA

### Читаем RUSLAN метадату

In [11]:
RUSLAN_METADATA = pd.read_csv(RUSLAN_METADATA_PATH, sep="|", header=None, index_col=0)
RUSLAN_METADATA.head()

,1
0,
000000_RUSLAN,С тревожным чувством берусь я за перо.
000001_RUSLAN,Кого интересуют признания литературного неудач...
000002_RUSLAN,Что поучительного в его исповеди?
000003_RUSLAN,Да и жизнь моя лишена внешнего трагизма.
000004_RUSLAN,Я абсолютно здоров.


### Для сокращений и аббревиатур будем использовать [ГПНТБ словарь](https://www.gpntb.ru/2017-08-21-14-53-51/339-produkty-gpntb/7630-elektronnyj-slovar-standartizovannykh-sokrashchenij.html)

In [12]:
SHORT_FORMS_ABBREVIATIONS = "data/short_forms.dict"

In [13]:
short_forms_abbreviations_df = pd.read_csv(SHORT_FORMS_ABBREVIATIONS)
# Убираем все, кроме пар {сокращение: слово/словосочетание}
short_forms_abbreviations_df = short_forms_abbreviations_df[
    ["Слово (словосочетание)", "Сокращение"]
]
short_forms_abbreviations_df = short_forms_abbreviations_df.dropna()

#### Нормализуем пробелы

In [14]:
def normalize_spaces(string):
    return re.sub(r'\s+', ' ', string).strip()

In [15]:
short_forms_abbreviations_df["Слово (словосочетание)"] = short_forms_abbreviations_df[
    "Слово (словосочетание)"
].apply(normalize_spaces)

short_forms_abbreviations_df["Сокращение"] = short_forms_abbreviations_df[
    "Сокращение"
].apply(normalize_spaces)

short_forms_abbreviations_df.head(5)

,Слово (словосочетание),Сокращение
0,например,напр.
1,сравнение; сравни,ср.
2,Германская Демократическая Республика,ГДР
3,Всемирный совет мира,ВСМ
4,Всемирная организация здравоохранения,ВОЗ


#### Разделяем сокращения и аббревиатуры

In [16]:
def separate_short_forms_abbreviations(df: pd.DataFrame):
    """
    Разделять по КАПСУ нет смысла, есть одиночные сокращения типа `С`.
    Ну и плюс, вдруг там будут какие-нибудь сокращения капсом лучше брать так: 
    все, что с точкой на конце – сокращения; все, что нет – аббревиатуры
    """
    short_forms = {}
    abbreviations = {}
    for row in df.iterrows():
        if "." in row[1]["Сокращение"]:
            short_forms[row[1]["Сокращение"]] = row[1]["Слово (словосочетание)"]
        else:
            abbreviations[row[1]["Сокращение"]] = row[1]["Слово (словосочетание)"]
    return short_forms, abbreviations

In [17]:
short_forms_dict, abbreviations_dict = separate_short_forms_abbreviations(
    short_forms_abbreviations_df
)

In [18]:
short_forms_list = short_forms_dict.keys()
abbreviations_list = abbreviations_dict.keys()

In [19]:
abbreviations_list

dict_keys(['ГДР', 'ВСМ', 'ВОЗ', 'США', 'ВФП', 'ВНР', 'ИСО', 'ИФЛА', 'ВМО', 'МОТ', 'ФАО', 'МАгАтЭ', 'МФД', 'НРБ', 'ООН', 'ЮНЕСКО', 'ЧССР', 'ПНР', 'ФРГ', 'ОЭСР', 'СРР', 'вуз', 'изд-во', 'б-чка', 'з-д', 'б-ка', 'С', 'ун-т', 'ин-т', 'км', 'г', 'З', 'кн-во', 'кВт', 'к/с', 'АО', 'хоз-во', 'ПВ', 'ч', 'р-н', 'пр-во', 'пром-сть', 'стр-во', 'ф-ка', 'сут', 'т-во', 'ст-ца', 'пр-ция', 'п-ов', 'мфише҆', '№', 'мин', 'м-во', 'м-б', 'нп', 'К°; Ко', 'Ю', 'гос-во', 'В', 'об-ние', 'о-во', 'о-ва', 'о-в', 'д-р', 'отд-ние', 'bokbd', 'bd', 'billedfonogr', 'AS', 'AD', 'AB', 'AG', 'Ko', 'ko', 'letop', 'ezds', 'FID', 'FIAB', 'dr', 'd-vo', 'ČSSR', 'cm', 'FSM', 'IAEA', 'afr', 'ca', 'hbd', 'hung', 'hullbd', 'indledn', 'ISO', 'ILO', 'IFLA', 'FAO', 'mficha', 'mfiche', 'mfișă', 'Mfiche', 'mforma', 'mfiš', 'mfisza', 'NV', 'MNK', 'mtárs', 'Mij', 'OMM', 'OECD', 'OIT', 'OY', 'ps', 'PRL', 'OCDE', 'UNESCO', 'OMS', 'OAA', 'ко', 'мфиш', 'obst', 'nr', 'no', 'obál', 'Obst', 'st', 'SA', 'RSR', 'ONU', 'sloven', 'wg', 'Ztg', 't-wo

### Считаем статистику по фразам в корпусе RUSLAN

#### Класс-счетчик

In [20]:
class EntryStatistics():
    def __init__(self, string):
        self.string = string
        self.string_preprocessed = self.preprocess(self.string)
        # self.string_tokenized = [token.text for token in tokenize(self.string)]
        self.string_tokenized = self.string.split()
        self.string_preprocessed_tokenized = self.string_preprocessed.split()

        # Приводим к Counter, потому что Counter'ы можно суммировать
        self.numerals = Counter(self.get_numerals())
        self.non_cyrillic = Counter(self.get_non_cyrillic())
        self.technical = Counter(self.get_technical())
        self.special_symbols = Counter(self.get_special_symbols())
        self.short_forms = Counter(self.get_short_forms())

        self.abbreviations = Counter(self.get_abbreviations())
        self.interjections = Counter(self.get_interjections())

    def preprocess(self, string):
        string = string.lower()
        return string

    def get_numerals(self):
        matches = re.findall(r'-?\d*\.?\d+', self.string)
        matches = [float(x) if '.' in x else int(x) for x in matches]
        matches_dict = dict.fromkeys(matches)
        for elem in matches_dict:
            matches_dict[elem] = matches.count(elem)
        return matches_dict

    def get_non_cyrillic(self):
        matches = [
            char
            for char in self.string
            if regex.fullmatch(r"[\p{L}]", char)
            and not regex.fullmatch(r"\p{Cyrillic}", char)
        ]
        matches_dict = dict.fromkeys(matches)
        for elem in matches_dict:
            matches_dict[elem] = matches.count(elem)
        return matches_dict

    def get_technical(self): # <-- воооооооооооооооооооот это доделать
        # 1) добавить игнорирование этих символов, если они в эмодзи
        # 2) добавить `-`, если он не дефис
        # 3) подумать над другими подобными случаями
        tech_symbols = r"[*|/@+<]"
        matches = re.findall(tech_symbols, self.string)
        matches_dict = dict.fromkeys(matches)
        for elem in matches_dict:
            matches_dict[elem] = matches.count(elem)
        return matches_dict
    
    def get_short_forms(self):
        tokens = self.string_tokenized
        short_forms_tokens = [
            (short_form, short_form.split())
            for short_form in short_forms_list
        ]
        matches = []

        i = 0
        while i < len(tokens):
            found = None
            found_length = 0
            for short_form, short_tokens in short_forms_tokens:
                if tokens[i:i + len(short_tokens)] == short_tokens:
                    if len(short_tokens) > found_length:
                        found = short_form
                        found_length = len(short_tokens)

            if found is not None:
                matches.append(found)
                i += found_length
            else:
                i += 1
        return dict(Counter(matches))

    def get_special_symbols(self):
        normal_symbols = '.,!?"\';…:-–«»-'
        matches = re.findall(rf"[^\w\s{normal_symbols}]", self.string)
        matches_dict = dict.fromkeys(matches)
        for elem in matches_dict:
            matches_dict[elem] = matches.count(elem)
        return matches_dict

    def get_punctuation(self):
        pass        

    def get_unusual_punctuation(self):
        pass

    def get_abbreviations(self):
        tokens = self.string_tokenized
        matches = []
        for abbreviation in abbreviations_list:
            if abbreviation.lower() in tokens:
                matches.append(abbreviation)
        matches_dict = dict.fromkeys(matches)
        for elem in matches_dict:
            matches_dict[elem] = matches.count(elem)
        return matches_dict

    def get_interjections(self):
        """
        Тут можно грузить nltk и определение частиречной принадлежности.
        Но если считать только междометия, то и просто список сгодится
        """
        with open("data/interjections.json", encoding="utf-8") as f:
            interjections = json.load(f)

        matches = []
        for interjection in interjections:
            if interjection.lower() in self.string_tokenized:
                matches.append(interjection)
        matches_dict = dict.fromkeys(matches)
        for elem in matches_dict:
            matches_dict[elem] = matches.count(elem)
        return matches_dict

    def get_emoji(self):
        matches = emoji.emoji_list(self.string)
        matches = [match["emoji"] for match in matches]
        matches_dict = dict.fromkeys(matches)
        for elem in matches_dict:
            matches_dict[elem] = matches.count(elem)

    def get_typos(self):
        pass

    def get_caps(self):
        pass

In [21]:
numerals = Counter()
non_cyrillic = Counter()
technical = Counter()
special_symbols = Counter()
short_forms = Counter()

abbreviations = Counter()
interjections = Counter()

In [22]:
for phrase in RUSLAN_METADATA.iterrows():
    entry_stats = EntryStatistics(phrase[1][1])

    numerals += entry_stats.numerals
    non_cyrillic += entry_stats.non_cyrillic
    technical += entry_stats.technical
    special_symbols += entry_stats.special_symbols
    short_forms += entry_stats.short_forms

    abbreviations += entry_stats.abbreviations
    interjections += entry_stats.interjections

#### Статистика

* топ-10 самых распространенных;
* встречаемость на символ/слово;
* встречаемость на фразу;
* 

##### Цифры

In [23]:
print(numerals) # type: ignore
print(non_cyrillic) # type: ignore
print(technical) # type: ignore
print(special_symbols) # type: ignore
print(short_forms) # type: ignore
print(abbreviations) # type: ignore # <- что-то тут странные аббревиатуры по типу "В", "С", "ко" и так далее
print(interjections) # type: ignore # <- почистить список междометий, чтобы туда не попадало "есть", "их"

Counter({61: 1, 129: 1, 16: 1, 6: 1})
Counter()
Counter({'/': 4, '*': 1, '<': 1})
Counter({'(': 305, ')': 305, '„': 72, '“': 57, '”': 13, '’': 6, '/': 4, '*': 1})
Counter({'лет.': 39, 'год.': 24, 'прав.': 10, 'час.': 10, 'слов.': 9, 'поэт.': 6, 'центр.': 5, 'М.': 5, 'договор.': 4, 'груз.': 4, 'пол.': 4, 'лингвист.': 3, 'Л.': 3, 'лист.': 3, 'нем.': 3, 'бар.': 3, 'фронт.': 3, 'зал.': 3, 'совет.': 2, 'пор.': 2, 'дух.': 2, 'марок.': 2, 'оркестр.': 2, 'район.': 2, 'демократ.': 2, 'сел.': 2, 'образ.': 2, 'черт.': 2, 'прим.': 1, 'цифр.': 1, 'правил.': 1, 'бот.': 1, 'ром.': 1, 'просмотр.': 1, 'состав.': 1, 'тип.': 1, 'т.': 1, 'д.': 1, 'о.': 1, 'букв.': 1, 'рос.': 1, 'мод.': 1, 'т. е.': 1, 'и т. д.': 1, 'период.': 1, 'рев.': 1})
Counter({'В': 4681, 'С': 1949, 'ко': 74, 'З': 1, 'сие': 1, 'АД': 1})
Counter({'и': 4953, 'у': 942, 'о': 638, 'а': 532, 'ее': 507, 'есть': 428, 'их': 275, 'но': 268, 'вот': 207, 'будет': 177, 'всего': 106, 'голос': 77, 'нее': 73, 'пока': 47, 'назад': 43, 'место': 41, 'ря

## Классификатор нормализован-не нормализован

### Иморты и константы

In [24]:
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [25]:
DEV_TRAIN_PATH = "data/dev_sentences_train.csv"
DEV_TEST_PATH = "data/dev_sentences_train.csv"

In [26]:
DEV_TRAIN = pd.read_csv(DEV_TRAIN_PATH, sep="|")
DEV_TEST = pd.read_csv(DEV_TEST_PATH, sep="|")

In [27]:
dev_sents_train = DEV_TRAIN["text"].to_list()
dev_labels_train = DEV_TRAIN["is_normalized"].to_list()

dev_sents_test = DEV_TEST["text"].to_list()
dev_labels_test = DEV_TEST["is_normalized"].to_list()

In [28]:
dev_sents_train[:5]

['Кстати, списание произойдёт в четыре часа дня по московскому времени.',
 'Светлана Юрьевна, раздел пятнадцатый договора описывает эту услугу.',
 'Комиссия составляет 15 процентов.',
 'Она перечитала письмо дважды, но так и не поняла, чего от неё хотят.',
 'Пётр Аркадьевич, пункт четвёртый договора уточняет условия.']

In [29]:
dev_labels_train[:5]

[1, 1, 0, 1, 1]

### multilingual-e5-large + классификатор

#### Эмбеддинги

In [30]:
def average_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

In [31]:
dev_sents_train = list(map(lambda x: f"query: {x}", dev_sents_train))
dev_sents_test = list(map(lambda x: f"query: {x}", dev_sents_test))

In [32]:
tokenizer = AutoTokenizer.from_pretrained(
    'intfloat/multilingual-e5-large',
    )
model = AutoModel.from_pretrained(
    'intfloat/multilingual-e5-large',
    torch_dtype=torch.float16,
    ).to(device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 391/391 [00:12<00:00, 31.65it/s]


In [33]:
def embed(sentences):
    # Токенизация
    batch_dict = tokenizer(
        sentences, max_length=512, padding=True, truncation=True, return_tensors='pt'
    )

    batch_dict = {
        k: v.to(device)
        for k, v in batch_dict.items()
    }

    with torch.inference_mode():
        outputs = model(**batch_dict)

        embeddings = average_pool(
            outputs.last_hidden_state, batch_dict['attention_mask']
        )

        # Нормализация эмбеддингов
        embeddings = F.normalize(embeddings, p=2, dim=1)

        # scores = (embeddings[:2] @ embeddings[2:].T) * 100
        # print(scores.tolist())
    return embeddings.to("cpu")

In [34]:
embeddings_train = []
embeddings_test = []

In [35]:
batch_size = 8

In [36]:
for i in tqdm(range(0, len(dev_sents_train), batch_size)):
    embeddings = embed(dev_sents_train[i : i + batch_size])
    embeddings_train.extend(embeddings)

100%|██████████| 125/125 [00:51<00:00,  2.41it/s]


In [37]:
for i in tqdm(range(0, len(dev_sents_test), batch_size)):
    embeddings = embed(dev_sents_test[i : i + batch_size])
    embeddings_test.extend(embeddings)

100%|██████████| 125/125 [00:51<00:00,  2.41it/s]


In [38]:
print(embeddings_train[0].shape)

torch.Size([1024])


In [ ]:
torch.save(dev_sents_train, "data/embeddings_weights/dev_sentences_emb_train.pt")
torch.save(dev_sents_test, "data/embeddings_weights/dev_sentences_emb_test.pt")

In [ ]:
torch.cuda.empty_cache()

#### Классификатор